# Overestimation bias in DDPG and TD3, with or without Layer Normalization

**Mini-project 1, UM5IN872 Reinforcement Learning.** Théo Fontaine, Théophile Lahaussois.

Questions of the study:

- **Q1.** Does Layer Normalization (LN) in the critic reduce the overestimation bias of DDPG and TD3?
- **Q2.** How does the overestimation bias relate to performance?

Protocol: a 2 × 2 design, {DDPG, TD3} × {critic without LN, critic with LN}. Each of the 4 configurations is trained with 5 seeds for 100k environment steps on `LunarLanderContinuous-v3`. Every 5k steps, we evaluate the deterministic policy on 10 episodes. The same episodes give the return and the bias, measured as $Q(s_t,a_t) - G_t$, where $G_t$ is the discounted Monte Carlo return.

This notebook contains the whole implementation and launches the runs, which are saved to `runs/`. `2_analysis.ipynb` reads `runs/` and produces the figures and statistics of the report.

Background: `notes/build/concepts.pdf`. The DDPG/TD3 code follows the course notebook `04-ddpg-td3.ipynb` and reuses the `rl_mind` building blocks.

## 1. Setup

In [ ]:
import copy
import json
import math
import os
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import gymnasium as gym
import torch
import torch.nn as nn
from torch import Tensor
from rl_mind.collectors import TransitionCollector
from rl_mind.core import Action, Actor
from rl_mind.data import ReplayBuffer, Transitions
from rl_mind.env import VecEnv
from rl_mind.nn import soft_update

RUNS = Path("runs")

### Hyper-parameters

They are all in one dataclass, so every run saves its exact configuration next to its results. The values are the usual TD3 values (Fujimoto et al., 2018), and they are the same for DDPG. The only differences between the configurations are:

- `algo`: `"ddpg"` gives 1 critic, no target policy smoothing and an actor update at every step. `"td3"` gives the 3 mechanisms of TD3: clipped double-Q, target policy smoothing, and delayed policy updates.
- `layernorm`: LayerNorm in the hidden layers of the critic(s).

Note: our DDPG keeps a target actor $\pi_{\bar\theta}$, as in the original paper and in Fujimoto et al.'s "Our DDPG". The course notebook uses the current actor instead. Keeping it in both algorithms means DDPG and TD3 differ *only* by the three TD3 mechanisms.

In [ ]:
@dataclass(frozen=True)
class Config:
    algo: str = "td3"          # "ddpg" or "td3"
    layernorm: bool = False    # LayerNorm in the critic's hidden layers
    seed: int = 0

    env_name: str = "LunarLanderContinuous-v3"
    max_steps: int = 100_000       # environment steps
    learning_starts: int = 10_000  # uniform random actions before learning
    buffer_size: int = 100_000     # = max_steps: nothing is ever discarded
    batch_size: int = 256
    gamma: float = 0.99
    tau: float = 0.005             # soft update of the target networks
    hidden: tuple[int, ...] = (256, 256)
    lr_actor: float = 1e-3
    lr_critic: float = 1e-3
    action_noise: float = 0.1      # std of the Gaussian exploration noise

    # TD3 only
    policy_delay: int = 2          # critic updates per actor update
    target_noise: float = 0.2      # target policy smoothing: noise std...
    target_noise_clip: float = 0.5 # ...and clipping

    eval_interval: int = 5_000
    n_eval_episodes: int = 10
    bias_tol: float = 0.01         # truncated episodes: see section 4

    @property
    def name(self) -> str:
        return self.algo + ("-ln" if self.layernorm else "")

    @property
    def run_dir(self) -> Path:
        return RUNS / self.name / f"seed{self.seed}"

## 2. Networks, with or without LayerNorm

The MLP is `rl_mind.nn.build_mlp` plus one switch: with `layernorm=True`, every *hidden* linear layer is followed by `nn.LayerNorm`, before the ReLU:

`Linear → LayerNorm → ReLU → Linear → LayerNorm → ReLU → Linear`

There is no LN after the output layer: $Q$ must be able to take any value.

**Why this can limit overestimation** (notes, section 4.2). LN normalizes the features of the last hidden layer, so for *any* input $|Q_\phi(s,a)| \le \|w\|\,(\|\gamma\|_\infty\sqrt d + \|\beta\|) + |b|$. Without LN, a ReLU network is piecewise linear and can extrapolate to arbitrarily large values far from the data, which the actor then exploits.

LN is used in the critic only: the critic is where the bias comes from, and leaving the actor unchanged keeps the comparison clean.

In [ ]:
def mlp(sizes: list[int], layernorm: bool = False, output_activation: nn.Module | None = None) -> nn.Sequential:
    layers: list[nn.Module] = []
    for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
        layers.append(nn.Linear(n_in, n_out))
        is_hidden = i < len(sizes) - 2
        if is_hidden:
            if layernorm:
                layers.append(nn.LayerNorm(n_out))
            layers.append(nn.ReLU())
        elif output_activation is not None:
            layers.append(output_activation)
    return nn.Sequential(*layers)


class Critic(nn.Module):
    # Q_phi(s, a): [B, obs_dim] x [B, action_dim] -> [B]
    def __init__(self, obs_dim: int, action_dim: int, hidden: tuple[int, ...], layernorm: bool):
        super().__init__()
        self.model = mlp([obs_dim + action_dim, *hidden, 1], layernorm=layernorm)

    def forward(self, obs: Tensor, action: Tensor) -> Tensor:
        return self.model(torch.cat([obs, action], dim=1)).squeeze(-1)


class DeterministicActor(Actor[Action]):
    # pi_theta(s) in [-1, 1]^d (final tanh); LunarLander's actions are in [-1, 1]^2
    def __init__(self, obs_dim: int, action_dim: int, hidden: tuple[int, ...]):
        super().__init__()
        self.model = mlp([obs_dim, *hidden, action_dim], output_activation=nn.Tanh())

    def forward(self, obs: Tensor) -> Action:
        return Action(value=self.model(obs))


class GaussianNoise(Actor[Action]):
    # Exploration at training time: a + N(0, sigma^2), clipped to the action range.
    # `act` (evaluation) stays the deterministic action of the wrapped actor.
    def __init__(self, actor: Actor[Action], sigma: float):
        super().__init__()
        self.actor, self.sigma = actor, sigma

    def forward(self, obs: Tensor) -> Action:
        a = self.actor(obs).value
        return Action(value=(a + self.sigma * torch.randn_like(a)).clamp(-1, 1))

    def act(self, obs: Tensor) -> Tensor:
        return self.actor.act(obs)


class UniformRandom(Actor[Action]):
    # Used during the first `learning_starts` steps, to fill the buffer with diverse data
    def __init__(self, action_dim: int):
        super().__init__()
        self.action_dim = action_dim

    def forward(self, obs: Tensor) -> Action:
        return Action(value=torch.rand(obs.shape[0], self.action_dim) * 2 - 1)

Quick check: where LN goes.

In [ ]:
print(mlp([10, 256, 256, 1], layernorm=False))
print(mlp([10, 256, 256, 1], layernorm=True))
assert [type(m) for m in mlp([3, 8, 4, 1], layernorm=True)] == [
    nn.Linear, nn.LayerNorm, nn.ReLU, nn.Linear, nn.LayerNorm, nn.ReLU, nn.Linear]
assert Critic(8, 2, (32, 32), True)(torch.randn(5, 8), torch.randn(5, 2)).shape == (5,)

## 3. DDPG and TD3: one update step

Course slides `ddpg.pdf`, slides 7–10 (DDPG) and 16–17 (TD3); notes, sections 1 and 3. One gradient step, given a batch $(s, a, r, s', d)$ from the replay buffer:

1. **Target actions.** $\tilde a' = \pi_{\bar\theta}(s')$ for DDPG. TD3 adds **target policy smoothing**: $\tilde a' = \mathrm{clip}(\pi_{\bar\theta}(s') + \mathrm{clip}(\epsilon, -c, c), -1, 1)$ with $\epsilon \sim \mathcal N(0, \sigma^2)$.
2. **TD target.** $y = r + \gamma (1-d) \min_i Q_{\bar\phi_i}(s', \tilde a')$. DDPG has a single critic, so the min disappears. TD3 takes the min of its two target critics (**clipped double-Q**).
   - Only `terminated` ($d$) stops the bootstrap. When the time limit truncates an episode, we still bootstrap (see the course notebook).
   - $y$ is computed under `torch.no_grad()`: it is a fixed regression target.
3. **Critic step.** Every critic regresses towards the *same* $y$: $L = \sum_i \mathrm{mean}_b (Q_{\phi_i}(s_b, a_b) - y_b)^2$. One Adam optimizer covers all the critics.
4. **Actor step and soft update.**
   - The actor step maximizes $Q_{\phi_1}(s, \pi_\theta(s))$, i.e. minimizes $-\mathrm{mean}_b\, Q_{\phi_1}(s_b, \pi_\theta(s_b))$. Only the actor optimizer steps. The critic receives gradients too, but they are cleared by `zero_grad` before its next step.
   - Then every target moves slowly: $\bar x \leftarrow \tau x + (1-\tau) \bar x$.
   - DDPG does this at every update. TD3 does it only every `policy_delay` critic updates (**delayed policy updates**, the *critic-to-actor ratio* of the brief).

In [ ]:
class Agent:
    def __init__(self, obs_dim: int, action_dim: int, cfg: Config):
        self.cfg = cfg
        self.is_td3 = cfg.algo == "td3"
        n_critics = 2 if self.is_td3 else 1

        self.actor = DeterministicActor(obs_dim, action_dim, cfg.hidden)
        self.critics = nn.ModuleList(
            Critic(obs_dim, action_dim, cfg.hidden, cfg.layernorm) for _ in range(n_critics))
        self.target_actor = copy.deepcopy(self.actor).requires_grad_(False)
        self.target_critics = copy.deepcopy(self.critics).requires_grad_(False)

        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=cfg.lr_actor)
        self.critic_optimizer = torch.optim.Adam(self.critics.parameters(), lr=cfg.lr_critic)
        self.n_updates = 0

    def update(self, batch: Transitions) -> dict[str, float]:
        cfg = self.cfg

        # 1-2. Target, without gradient
        with torch.no_grad():
            next_action = self.target_actor(batch.next_obs).value
            if self.is_td3:  # target policy smoothing
                noise = (cfg.target_noise * torch.randn_like(next_action)).clamp(
                    -cfg.target_noise_clip, cfg.target_noise_clip)
                next_action = (next_action + noise).clamp(-1, 1)
            next_q = torch.stack([tc(batch.next_obs, next_action) for tc in self.target_critics])  # [C, B]
            next_q = next_q.min(dim=0).values  # clipped double-Q (identity with 1 critic)
            target = batch.reward + cfg.gamma * (~batch.terminated).float() * next_q  # [B]

        # 3. Critic step: every critic towards the same target
        q = torch.stack([c(batch.obs, batch.action.value) for c in self.critics])  # [C, B]
        critic_loss = ((q - target) ** 2).mean(dim=1).sum()
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        self.n_updates += 1
        out = {"critic_loss": critic_loss.item()}

        # 4. Actor step and soft updates, delayed for TD3
        delay = cfg.policy_delay if self.is_td3 else 1
        if self.n_updates % delay == 0:
            actor_loss = -self.critics[0](batch.obs, self.actor(batch.obs).value).mean()
            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()
            soft_update(self.actor, self.target_actor, cfg.tau)
            soft_update(self.critics, self.target_critics, cfg.tau)
            out["actor_loss"] = actor_loss.item()
        return out

## 4. Measuring the overestimation bias

Notes, section 5. We run the **deterministic** policy $\pi_\theta$ (no exploration noise) for `n_eval_episodes` episodes. For each visited pair $(s_t, a_t)$, we compute the discounted Monte Carlo return, backwards:

$$G_t = r_{t+1} + \gamma\, G_{t+1}, \qquad G_{T-1} = r_T .$$

$G_t$ is an unbiased estimate of $Q^\pi(s_t, a_t)$, and we compare it with $Q_{\phi_1}(s_t, a_t)$, the critic the actor maximizes.

- **Normalized bias** (Chen et al., 2021, REDQ): $\delta = (Q - G) / |\overline{G}|$. We report its mean (> 0 means overestimation) and its std. We also keep the raw $\overline{Q - G}$.
- **Truncation.** LunarLander cuts episodes at 1000 steps. For a *truncated* episode, $G_t$ misses everything after the cut, and that tail weighs $\gamma^{T-t}$. We keep only the steps with $\gamma^{T-t} \le$ `bias_tol`, i.e. we drop the last $H = \lceil \ln(\text{tol})/\ln\gamma \rceil = 459$ steps. For a *terminated* episode (landing or crash), the future value really is 0, so every step is kept.
- **Pitfall.** $Q$ is a *discounted* quantity. It is compared with $G_t$, not with the undiscounted episode return.
- **Episode seeds.** Episode $i$ of every evaluation uses the seed `10_000 + 100·seed + i`, so all the evaluations of a run start from the same 10 initial states. This reduces the noise between consecutive evaluations.

In [ ]:
def discounted_returns(rewards: Tensor, gamma: float) -> Tensor:
    g = torch.zeros_like(rewards)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running
        g[t] = running
    return g


def valid_length(length: int, truncated: bool, gamma: float, tol: float) -> int:
    if not truncated:
        return length
    horizon = math.ceil(math.log(tol) / math.log(gamma))
    return max(length - horizon, 0)


@torch.no_grad()
def evaluate(agent: Agent, cfg: Config) -> dict:
    env = gym.make(cfg.env_name)
    obs_list, act_list, g_list, returns, lengths = [], [], [], [], []
    for i in range(cfg.n_eval_episodes):
        obs, _ = env.reset(seed=10_000 + 100 * cfg.seed + i)
        ep_obs, ep_act, ep_rew = [], [], []
        terminated = truncated = False
        while not (terminated or truncated):
            obs_t = torch.as_tensor(obs, dtype=torch.float32)
            action = agent.actor.act(obs_t[None])[0]
            obs, reward, terminated, truncated, _ = env.step(action.numpy())
            ep_obs.append(obs_t); ep_act.append(action); ep_rew.append(float(reward))
        rewards = torch.tensor(ep_rew)
        returns.append(float(rewards.sum()))
        lengths.append(len(rewards))
        keep = valid_length(len(rewards), truncated, cfg.gamma, cfg.bias_tol)
        if keep > 0:
            obs_list.append(torch.stack(ep_obs[:keep]))
            act_list.append(torch.stack(ep_act[:keep]))
            g_list.append(discounted_returns(rewards, cfg.gamma)[:keep])
    env.close()

    obs, act, g = torch.cat(obs_list), torch.cat(act_list), torch.cat(g_list)
    q = agent.critics[0](obs, act)
    delta = (q - g) / g.mean().abs()
    return {
        "returns": returns,
        "return_mean": sum(returns) / len(returns),
        "bias_mean": float(delta.mean()),
        "bias_std": float(delta.std(unbiased=False)),
        "raw_bias": float((q - g).mean()),
        "q_mean": float(q.mean()),
        "g_mean": float(g.mean()),
        "n_samples": len(g),
        "length_mean": sum(lengths) / len(lengths),
    }

In [ ]:
# Checks on small hand-computed cases
assert torch.allclose(discounted_returns(torch.tensor([1.0, 1.0, 1.0]), 0.5), torch.tensor([1.75, 1.5, 1.0]))
assert valid_length(1000, truncated=True, gamma=0.99, tol=0.01) == 1000 - 459
assert valid_length(300, truncated=False, gamma=0.99, tol=0.01) == 300

## 5. Training loop

As in the course notebook:

- `learning_starts` steps of uniform random actions, then the actor with Gaussian noise.
- One gradient step per collected transition (UTD ratio = 1).
- An evaluation every `eval_interval` steps.

After each evaluation, the results are written to `runs/<config>/seed<k>/metrics.json`. A finished run is not run again, so the grid can be interrupted and relaunched.

In [ ]:
def train(cfg: Config, verbose: bool = False) -> dict:
    path = cfg.run_dir / "metrics.json"
    if path.exists() and json.loads(path.read_text()).get("done"):
        return json.loads(path.read_text())
    cfg.run_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(cfg.seed)
    env = VecEnv(cfg.env_name, 1, seed=cfg.seed)
    agent = Agent(env.observation_dim, env.action_dim, cfg)
    collector = TransitionCollector(env, UniformRandom(env.action_dim))
    buffer = ReplayBuffer(cfg.buffer_size)

    metrics = {"config": asdict(cfg), "name": cfg.name, "evals": []}
    losses: dict[str, list[float]] = {}
    next_eval, start = cfg.eval_interval, time.perf_counter()

    while collector.steps < cfg.max_steps:
        if collector.steps >= cfg.learning_starts and isinstance(collector.actor, UniformRandom):
            collector.actor = GaussianNoise(agent.actor, cfg.action_noise)
        buffer.add(collector.collect(1))

        if collector.steps >= cfg.learning_starts:
            for k, v in agent.update(buffer.sample(cfg.batch_size)).items():
                losses.setdefault(k, []).append(v)

        if collector.steps >= next_eval:
            next_eval += cfg.eval_interval
            record = {"step": collector.steps, "time": time.perf_counter() - start, **evaluate(agent, cfg),
                      **{f"train_{k}": sum(v) / len(v) for k, v in losses.items()}}
            losses.clear()
            metrics["evals"].append(record)
            path.write_text(json.dumps(metrics))
            if verbose:
                print(f"step {record['step']:>6}  return {record['return_mean']:7.1f}  "
                      f"bias {record['bias_mean']:+.3f}  ({record['time']:.0f}s)")

    metrics["done"] = True
    path.write_text(json.dumps(metrics))
    return metrics

## 6. Sanity check: a short TD3 run

A few thousand steps, to check that everything runs before launching the grid. It doesn't learn anything yet: the bias at this stage mostly reflects an untrained critic.

In [ ]:
quick = Config(algo="td3", layernorm=True, max_steps=3_000, learning_starts=1_000, eval_interval=1_000,
               n_eval_episodes=2)
RUNS_MAIN, RUNS = RUNS, Path("runs_quick")  # don't pollute runs/
train(quick, verbose=True)
RUNS = RUNS_MAIN

## 7. The grid: 4 configurations × 5 seeds, in parallel

`joblib` runs the trainings in 4 separate processes, one per physical core, each limited to 1 torch thread. On our laptop (i5-10210U, CPU only), one run takes about 35 min, so the whole grid takes about 3 h. The seeds are interleaved, so that partial results are always balanced across configurations.

Progress shows up below, and also in `runs/*/seed*/metrics.json`, which `2_analysis.ipynb` can read while the runs are still going. To relaunch after an interruption, re-run the cell: finished runs are skipped.

In [ ]:
from joblib import Parallel, delayed

SEEDS = range(5)
GRID = [Config(algo=algo, layernorm=ln, seed=seed)
        for seed in SEEDS for algo in ["ddpg", "td3"] for ln in [False, True]]


def run_one(cfg: Config) -> str:
    torch.set_num_threads(1)
    t = time.perf_counter()
    m = train(cfg)
    return f"{cfg.name} seed{cfg.seed}: final return {m['evals'][-1]['return_mean']:.1f} ({(time.perf_counter() - t) / 60:.0f} min)"


RUN_GRID = True
if RUN_GRID:
    for line in Parallel(n_jobs=4, return_as="generator_unordered")(delayed(run_one)(c) for c in GRID):
        print(line, flush=True)